# 도구 호출 해 보기 · OpenAI 형식

강의 20 ~ 26장에서 본 것을 실제 호출로 확인한다. `llm-call.ipynb` 와 같은 환경에서 돈다.

- LLM 에게 도구를 알려 주면, LLM 은 도구를 **부르겠다는 요청**만 돌려준다
- 실제 실행은 우리 코드가 하고, 결과를 다시 `messages` 에 넣어 보낸다

In [1]:
// 접속 정보: 이 세 값만 바꾸면 OpenAI, Ollama, vLLM 어디든 같은 코드로 돈다
const BASE_URL = Deno.env.get("LLM_BASE_URL") ?? "http://localhost:11434/v1";  // OpenAI: https://api.openai.com/v1
const API_KEY  = Deno.env.get("LLM_API_KEY")  ?? "ollama";                     // Ollama 는 아무 값이나 된다
const MODEL    = Deno.env.get("LLM_MODEL")    ?? "qwen3.5:9b-q4_K_M";

// qwen3.5 처럼 답하기 전에 "생각" 부터 하는 모델은, 시연에서 답만 바로 보려고 생각 단계를 끈다
// 이 옵션을 받지 않는 모델(예: OpenAI gpt-4o 계열)이면 {} 로 바꾼다
// temperature 0: 시연이 매번 같게 나오도록. 높이면 가끔 도구를 부르지 않고 답을 지어낸다
const OPTIONS = { reasoning_effort: "none", temperature: 0 };

앞 노트북의 `chat()` 과 같다. 이번에는 메시지에 도구 호출이 끼어들어서 모양이 조금 넓어졌다.

In [2]:
type ToolCall = { id: string; type: "function"; function: { name: string; arguments: string } };
type Message = {
  role: "system" | "user" | "assistant" | "tool";
  content: string | null;
  tool_calls?: ToolCall[];   // assistant 가 도구를 부를 때
  tool_call_id?: string;     // tool 메시지가 어느 호출의 결과인지
  reasoning?: string;        // 생각 단계를 켜면 Ollama 가 함께 돌려주는 생각
};

async function chat(messages: Message[], extra: Record<string, unknown> = {}) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, ...OPTIONS, ...extra }),
  });
  if (!res.ok) throw new Error(`${res.status} ${await res.text()}`);
  return await res.json();
}

function show(messages: Message[]) {
  for (const m of messages) {
    const body = m.tool_calls
      ? m.tool_calls.map((c) => `🔧 ${c.function.name}(${c.function.arguments})`).join(", ")
      : m.content;
    console.log(`  ${m.role.padEnd(9)} │ ${body}`);
  }
}

## ① LLM 에게 도구를 알려 준다 (21장)

도구 설명은 요청의 `tools` 에 담는다. 21장의 세 가지가 그대로 들어간다.

- `name`: 이름
- `description`: 언제 쓰는지
- `parameters`: 어떻게 부르는지. 넣을 값의 이름과 모양

In [3]:
const tools = [{
  type: "function",
  function: {
    name: "get_schedule",                                                     // 이름
    description: "교육장의 예약 현황을 조회한다. 교육장 일정을 알아야 할 때 쓴다.",  // 언제
    parameters: {                                                             // 어떻게
      type: "object",
      properties: {
        room: { type: "string", description: "교육장 이름. 예: 제2교육장" },
        date: { type: "string", description: "날짜. YYYY-MM-DD" },
      },
      required: ["room", "date"],
    },
  },
}];

const SYSTEM: Message = { role: "system", content: "너는 교육장 일정 담당이다. 오늘은 2026-10-20 이다. 짧게 답한다." };

### 도구가 필요한 질문

In [4]:
const messages: Message[] = [SYSTEM, { role: "user", content: "29일에 제2교육장 비어 있어?" }];

const r1 = await chat(messages, { tools });
const call: Message = r1.choices[0].message;
console.log("finish_reason:", r1.choices[0].finish_reason);
console.log(JSON.stringify(call, null, 2));

finish_reason: tool_calls
{
  "role": "assistant",
  "content": "",
  "tool_calls": [
    {
      "id": "call_m9x90w13",
      "index": 0,
      "type": "function",
      "function": {
        "name": "get_schedule",
        "arguments": "{\"room\":\"제2교육장\",\"date\":\"2026-10-29\"}"
      }
    }
  ]
}


- `content` 는 비어 있고, 대신 `tool_calls` 에 **어떤 도구를, 어떤 값으로** 부를지가 들어 있다
- `finish_reason` 도 `"tool_calls"` 다. "답이 아니라 도구 요청이다" 라는 뜻이다
- LLM 은 아무것도 실행하지 않았다. 요청을 글로 썼을 뿐이다. `arguments` 도 그냥 JSON 문자열이다

### 도구가 필요 없는 질문

같은 도구를 줘도, 필요 없으면 부르지 않는다. 부를지 말지도 LLM 이 고른다.

In [5]:
const r0 = await chat([SYSTEM, { role: "user", content: "교육장 예약은 보통 며칠 전에 하는 게 좋아?" }], { tools });
console.log("finish_reason:", r0.choices[0].finish_reason);
console.log("tool_calls:", r0.choices[0].message.tool_calls ?? "없음");
console.log("답:", r0.choices[0].message.content);

finish_reason: stop
tool_calls: 없음
답: 보통 1~2 주 전이 적당합니다. 인기 있는 교육장은 더 일찍 예약해야 할 수도 있으니, 꼭 필요한 날짜의 일정을 미리 확인하시는 게 좋습니다.


## ② 실행은 우리 책임이다 (21장)

진짜 도구는 우리 코드에 있는 평범한 함수다. 여기서는 예약 장부를 흉내 낸다.

In [6]:
const BOOKINGS: Record<string, Record<string, string[]>> = {
  "제2교육장": { "2026-10-29": ["09:00-12:00 신입 교육", "13:00-16:00 보안 교육"] },
};

function getSchedule({ room, date }: { room: string; date: string }) {
  return { room, date, booked: BOOKINGS[room]?.[date] ?? [] };
}

LLM 이 쓴 요청을 읽어서, 우리가 함수를 부른다. 결과는 `tool` 메시지로 배열에 넣는다. `tool_call_id` 로 어느 요청의 결과인지 짝을 맞춘다.

In [7]:
const withResult: Message[] = [...messages, call];   // LLM 의 도구 요청도 대화의 일부다

for (const tc of call.tool_calls!) {
  const args = JSON.parse(tc.function.arguments);
  console.log("우리가 실행:", tc.function.name, args);
  const result = getSchedule(args);                  // ← 실행은 여기, 우리 코드에서
  withResult.push({ role: "tool", tool_call_id: tc.id, content: JSON.stringify(result) });
}

우리가 실행: get_schedule { room: "제2교육장", date: "2026-10-29" }


4

결과를 넣은 배열로 한 번 더 부른다.

In [8]:
show(withResult);

const r2 = await chat(withResult, { tools });
console.log("\n답:", r2.choices[0].message.content);

  system    │ 너는 교육장 일정 담당이다. 오늘은 2026-10-20 이다. 짧게 답한다.
  user      │ 29일에 제2교육장 비어 있어?
  assistant │ 🔧 get_schedule({"room":"제2교육장","date":"2026-10-29"})
  tool      │ {"room":"제2교육장","date":"2026-10-29","booked":["09:00-12:00 신입 교육","13:00-16:00 보안 교육"]}

답: 아니요, 비어있지 않습니다. 09:00~12:00에 '신입 교육', 13:00~16:00에 '보안 교육'이 예약되어 있습니다.


LLM 은 두 번 불렸다. 한 번은 도구를 고르려고, 한 번은 결과를 보고 답하려고. 그 사이에서 실행한 건 우리 코드다.

## ③ LLM 은 결과가 진짜인지 모른다

이번에는 함수를 실행하지 않고, 결과를 **지어내서** 넣어 본다.

In [9]:
const fake: Message[] = [
  ...messages,
  call,
  {
    role: "tool",
    tool_call_id: call.tool_calls![0].id,
    content: JSON.stringify({ room: "제2교육장", date: "2026-10-29", booked: ["00:00-24:00 공사로 사용 불가"] }),
  },
];

const r3 = await chat(fake, { tools });
console.log("답:", r3.choices[0].message.content);

답: 아니요, 29일에는 공사 중이라 예약이 불가능합니다.


LLM 은 `messages` 에 적힌 대로 믿는다. 도구를 실제로 실행했는지, 결과가 맞는지 확인할 방법이 없다. 그래서 무엇을 실행하고 무엇을 돌려줄지는 하네스가 책임져야 한다.

## ④ 루프로 감싸기 (24 ~ 26장)

지금까지 한 일은 "도구 요청 → 실행 → 결과를 넣고 다시 호출" 한 바퀴였다. 이걸 `while` 로 감싸면 Agent 가 된다.

이번에는 도구를 두 개 준다. 두 번째 도구는 첫 번째 결과를 봐야 부를 수 있다.

도구를 여러 번 고르는 일이라, 여기서는 모델의 생각 단계를 다시 켠다. 그러면 모델이 무엇을 판단했는지(Thought)도 함께 볼 수 있다.

In [10]:
const loopTools = [
  {
    type: "function",
    function: {
      name: "find_free_rooms",
      description: "날짜에 비어 있는 교육장과 빈 시간대를 찾는다.",
      parameters: {
        type: "object",
        properties: {
          date: { type: "string", description: "날짜. YYYY-MM-DD" },
          hours: { type: "number", description: "필요한 시간 수" },
        },
        required: ["date", "hours"],
      },
    },
  },
  {
    type: "function",
    function: {
      name: "get_maintenance",
      description: "교육장의 장비 점검 일정을 조회한다. 점검 시간에는 그 교육장을 쓸 수 없다.",
      parameters: {
        type: "object",
        properties: {
          room: { type: "string", description: "교육장 이름" },
          date: { type: "string", description: "날짜. YYYY-MM-DD" },
        },
        required: ["room", "date"],
      },
    },
  },
];

// 진짜 도구: 우리 코드의 함수들. 여기서는 장부를 흉내 낸다
const ROOMS = ["세미나실 B-204"];

const toolImpl: Record<string, (args: any) => unknown> = {
  find_free_rooms: ({ date }) => (date === "2026-10-29" ? [{ room: "세미나실 B-204", free: "14:00-18:00" }] : []),
  get_maintenance: ({ room, date }) => {
    if (!ROOMS.includes(room)) return { error: `'${room}' 이라는 교육장은 없다` };
    return date === "2026-10-29" ? ["15:00-16:00 프로젝터 점검"] : [];
  },
};

루프가 하네스다. 끝나는 길은 두 가지다.

- ① 모델이 "끝": 도구 요청 없이 답이 오면 멈춘다
- ② 하네스가 "그만": 정해 둔 바퀴 수를 넘으면 멈춘다

In [11]:
async function runAgent(question: string, maxSteps = 5) {
  const messages: Message[] = [
    { role: "system", content: "너는 교육장 예약 담당이다. 오늘은 2026-10-20 이다. 장비 점검 시간을 피해서 추천하고, 교육장과 시간을 함께 알려 준다." },
    { role: "user", content: question },
  ];

  let steps = 0;
  while (true) {
    // 이번에는 생각 단계를 켠다(reasoning_effort 를 뺀다). 모델이 다음 행동을 더 잘 고르고, Thought 도 볼 수 있다
    const r = await chat(messages, { tools: loopTools, reasoning_effort: undefined });
    const msg: Message = r.choices[0].message;
    messages.push(msg);

    if (!msg.tool_calls?.length) {                      // ① 모델이 "끝"
      console.log(`\nFinal Answer  ${msg.content}`);
      return;
    }

    steps++;
    console.log(`── ${steps} 바퀴`);
    if (msg.reasoning) console.log(`  Thought      ${msg.reasoning.trim().replace(/\s+/g, " ")}`);
    for (const tc of msg.tool_calls) {
      const run = toolImpl[tc.function.name];
      const result = run ? run(JSON.parse(tc.function.arguments)) : { error: "없는 도구" };  // 에러도 결과로 돌려준다
      console.log(`  Action       ${tc.function.name}(${tc.function.arguments})`);
      console.log(`  Observation  ${JSON.stringify(result)}`);
      messages.push({ role: "tool", tool_call_id: tc.id, content: JSON.stringify(result) });
    }

    if (steps >= maxSteps) {                            // ② 하네스가 "그만"
      console.log(`\n${maxSteps} 바퀴 한도에 걸려 하네스가 멈췄다`);
      return;
    }
  }
}

### 끝까지 돌리기

In [12]:
await runAgent("29일에 2시간짜리 교육 잡을 수 있는 교육장 있어?");

── 1 바퀴
  Thought      사용자가 2026-10-29에 2시간짜리 교육을 예약할 수 있는 교육장을 찾고 있습니다. 먼저 find_free_rooms 함수를 호출해서 해당 날짜와 시간대에 비어 있는 교육장을 찾아야 합니다. 파라미터: - date: "2026-10-29" - hours: 2 함수를 호출한 후 결과를 바탕으로 답변을 작성하겠습니다.
  Action       find_free_rooms({"date":"2026-10-29","hours":2})
  Observation  [{"room":"세미나실 B-204","free":"14:00-18:00"}]
── 2 바퀴
  Thought      find_free_rooms 함수를 호출한 결과, 2026-10-29에 세미나실 B-204에서 14:00-18:00时间段이 비어있다는 결과가 나왔습니다. 사용자가 요청한 대로 장비 점검 시간을 피해서 추천해야 하므로, get_maintenance 함수를 호출해서 해당 날짜와 교육장의 장비 점검 일정을 확인해야 합니다.
  Action       get_maintenance({"room":"세미나실 B-204","date":"2026-10-29"})
  Observation  ["15:00-16:00 프로젝터 점검"]

Final Answer  네, 2026-10-29에 **세미나실 B-204**에서 교육이 가능합니다!

**사용 가능 시간:** 16:00 - 18:00 (2시간)

**참고사항:**
- 세미나실 B-204는 15:00-16:00에 장비 점검(프로젝터 점검)이 예정되어 있습니다.
- 따라서 14:00-18:00 중 15:00-16:00은 점검 시간으로 사용 불가합니다.
- 2시간 교육으로는 **16:00-18:00** 시간대가 적합합니다.

예약하시겠습니까?


- Thought · Action · Observation 이 바퀴마다 찍힌다
- 1 바퀴에서 빈 교육장을 찾고, 그 결과를 보고 나서야 2 바퀴에서 **그 교육장의** 점검 일정을 조회한다. 두 번째 도구는 처음부터 정할 수 없었다
- 더 부를 도구가 없다고 판단하면 모델이 답을 낸다. ① 모델이 "끝"

### 한도를 1 바퀴로 줄이면

In [13]:
await runAgent("29일에 2시간짜리 교육 잡을 수 있는 교육장 있어?", 1);

── 1 바퀴
  Thought      사용자가 2026-10-29에 2시간짜리 교육을 예약할 수 있는 교육장을 찾고 있습니다. 먼저 find_free_rooms 함수를 호출해서 해당 날짜와 시간대에 비어 있는 교육장을 찾아야 합니다. 파라미터: - date: "2026-10-29" - hours: 2 함수를 호출한 후 결과를 바탕으로 답변을 작성하겠습니다.
  Action       find_free_rooms({"date":"2026-10-29","hours":2})
  Observation  [{"room":"세미나실 B-204","free":"14:00-18:00"}]

1 바퀴 한도에 걸려 하네스가 멈췄다


모델은 더 하고 싶었지만 하네스가 끊었다. ② 하네스가 "그만". 한도가 없으면 모델이 같은 도구를 끝없이 부를 수도 있다. 루프를 돌리는 것도, 멈추는 것도 우리 코드다.

## 정리

- 도구는 `tools` 에 이름 · 언제 · 어떻게를 적어 알려 준다
- LLM 은 `tool_calls` 로 **요청만** 한다. 부를지 말지, 무엇을 넣을지를 고를 뿐이다
- 실행은 우리 코드가 하고, 결과를 `tool` 메시지로 배열에 넣어 다시 부른다
- LLM 은 받은 결과를 그대로 믿는다. 진짜 결과를 넣는 건 하네스의 몫이다
- 이걸 `while` 로 감싸면 Agent 루프가 된다. 루프를 돌리고 멈추는 것도 하네스다